Feature Store lab with experiment-based feature set comparison
*Co-authored with CoCo*

# E-Commerce Personalization: Feature Store Lab

**Objective:** Build an end-to-end ML pipeline for product purchase prediction using
the Snowflake Feature Store, Model Registry, and Online Serving.

**Domain:** E-commerce personalization — predicting whether a customer will purchase
a product (binary classification for recommendation scoring).

**What you'll learn:**
1. Define entities and register feature views in the Feature Store
2. Engineer derived features with Snowpark window functions
3. Generate point-in-time correct training datasets
4. Train and register an XGBoost classifier
5. Enable online serving for real-time feature retrieval
6. Deploy a model with automatic feature lookup

**Prerequisites:**
- `setup.sql` has been executed (creates database, schemas, and synthetic data)
- Database: `FEATURE_STORE_DEMO` with schemas: RAW, FEATURE_STORE, MODELS, SCORING
- Snowflake ML packages installed in the notebook environment

## 1. Setup & Imports

In [ ]:
# Core Snowpark
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StructType, StructField, StringType, FloatType
from snowflake.snowpark.window import Window

# Feature Store
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity
from snowflake.ml.feature_store.feature_view import OnlineConfig
from snowflake.ml.feature_store import StoreType

# Model Registry
from snowflake.ml.registry import Registry

# Modeling
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score
)

# Standard
import pandas as pd
import numpy as np
import time
from datetime import datetime

In [ ]:
# Get the active session (Snowflake Notebook provides this automatically)
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Configuration
DATABASE = "FEATURE_STORE_DEMO"
SCHEMA_RAW = "RAW"
SCHEMA_FS = "FEATURE_STORE"
SCHEMA_MODELS = "MODELS"
SCHEMA_SCORING = "SCORING"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA_RAW}").collect()

print(f"Connected as: {session.get_current_user()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {DATABASE}")

## 2. Explore Source Data

The `setup.sql` script created synthetic e-commerce data across several tables:

| Table | Description |
|-------|-------------|
| `CUSTOMERS` | Customer profiles: demographics, loyalty tier, account age |
| `PRODUCTS` | Product catalog: category, price, ratings, inventory |
| `PURCHASE_HISTORY` | Transaction log: who bought what, when, at what price |
| `BROWSE_EVENTS` | Clickstream: product views, cart adds, wishlist actions |
| `PRODUCT_DAILY_METRICS` | Aggregated daily product performance metrics |

Let's verify the data is loaded and explore its shape.

In [ ]:
# Check row counts for all source tables
tables = ["CUSTOMER_PROFILE", "PRODUCT_CATALOG", "PURCHASE_HISTORY", "BROWSE_EVENTS", "PRODUCT_DAILY_METRICS"]

print(f"{'Table':<25} {'Row Count':>12}")
print("-" * 40)
for table in tables:
    count = session.table(f"{DATABASE}.{SCHEMA_RAW}.{table}").count()
    print(f"{table:<25} {count:>12,}")

print("\n--- Sample: CUSTOMER_PROFILE ---")
session.table(f"{DATABASE}.{SCHEMA_RAW}.CUSTOMER_PROFILE").limit(5).show()

print("\n--- Sample: PRODUCT_CATALOG ---")
session.table(f"{DATABASE}.{SCHEMA_RAW}.PRODUCT_CATALOG").limit(5).show()

print("\n--- Sample: PURCHASE_HISTORY ---")
session.table(f"{DATABASE}.{SCHEMA_RAW}.PURCHASE_HISTORY").limit(5).show()

## 3. Feature Store Initialization

The Snowflake Feature Store organizes features around **Entities** — the real-world
objects that features describe. Each entity has one or more **join keys** that
uniquely identify instances.

For our e-commerce domain:
- **CUSTOMER** entity (join key: `CUSTOMER_ID`) — features about shoppers
- **PRODUCT** entity (join key: `PRODUCT_ID`) — features about items

Feature Views are then registered against these entities, enabling automatic
point-in-time joins during training set generation.

In [ ]:
# Initialize the Feature Store
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA_FS,
    default_warehouse=session.get_current_warehouse(),
    creation_mode="CREATE_IF_NOT_EXISTS",
)

# Define entities
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="E-commerce customer identified by unique customer ID",
)

product_entity = Entity(
    name="PRODUCT",
    join_keys=["PRODUCT_ID"],
    desc="Product in the catalog identified by unique product ID",
)

# Register entities
fs.register_entity(customer_entity)
fs.register_entity(product_entity)

print("Registered entities:")
fs.list_entities().show()

## 4. Register Raw Feature Views

Raw Feature Views expose source table columns as governed, discoverable features.
They define:
- Which **entity** the features belong to
- Which **columns** are features vs. keys
- An optional **timestamp column** for point-in-time correctness

We'll register three raw feature views from our source tables.

### 4.1 Customer Profile Features

In [ ]:
# Customer demographics, loyalty tier, and spend metrics
customer_profile_df = session.table(f"{DATABASE}.{SCHEMA_RAW}.CUSTOMER_PROFILE").select(
    "CUSTOMER_ID",
    "AGE_GROUP",
    "GENDER",
    "STATE",
    "LOYALTY_TIER",
    "DAYS_SINCE_LAST_ORDER",
    "LIFETIME_SPEND",
    "LIFETIME_ORDERS",
    "AVG_ORDER_VALUE",
    "PREFERRED_CATEGORY",
)

customer_profile_fv = FeatureView(
    name="CUSTOMER_PROFILE_FEATURES",
    entities=[customer_entity],
    feature_df=customer_profile_df,
    desc="Customer demographics, loyalty status, and lifetime spend metrics",
)
customer_profile_fv = fs.register_feature_view(
    feature_view=customer_profile_fv, version="v1", overwrite=True
)
print(f"Registered: {customer_profile_fv.name}/{customer_profile_fv.version}")
print(f"  Features: {[c for c in customer_profile_df.columns if c != 'CUSTOMER_ID']}")

### 4.2 Product Catalog Features

In [ ]:
# Product attributes: category, pricing, ratings
product_catalog_df = session.table(f"{DATABASE}.{SCHEMA_RAW}.PRODUCT_CATALOG").select(
    "PRODUCT_ID",
    "CATEGORY",
    "SUBCATEGORY",
    "PRICE",
    "AVG_RATING",
    "REVIEW_COUNT",
    "BRAND",
    "IN_STOCK",
)

product_catalog_fv = FeatureView(
    name="PRODUCT_CATALOG_FEATURES",
    entities=[product_entity],
    feature_df=product_catalog_df,
    desc="Product catalog attributes: category, pricing, and customer ratings",
)
product_catalog_fv = fs.register_feature_view(
    feature_view=product_catalog_fv, version="v1", overwrite=True
)
print(f"Registered: {product_catalog_fv.name}/{product_catalog_fv.version}")
print(f"  Features: {[c for c in product_catalog_df.columns if c != 'PRODUCT_ID']}")

### 4.3 Purchase History Features

In [ ]:
# Time-stamped purchase aggregates per customer
# This view has a timestamp_col for point-in-time correct joins
purchase_history_df = session.sql(f"""
    SELECT
        CUSTOMER_ID,
        PURCHASE_TS,
        PRODUCT_ID,
        QUANTITY,
        UNIT_PRICE,
        DISCOUNT_PCT,
        TOTAL_AMOUNT
    FROM {DATABASE}.{SCHEMA_RAW}.PURCHASE_HISTORY
""")

purchase_history_fv = FeatureView(
    name="PURCHASE_HISTORY_FEATURES",
    entities=[customer_entity],
    feature_df=purchase_history_df,
    timestamp_col="PURCHASE_TS",
    desc="Raw purchase transactions with timestamp for point-in-time joins",
)
purchase_history_fv = fs.register_feature_view(
    feature_view=purchase_history_fv, version="v1", overwrite=True
)
print(f"Registered: {purchase_history_fv.name}/{purchase_history_fv.version}")
print(f"  Timestamp column: PURCHASE_TS (enables point-in-time correctness)")

**Checkpoint:** Three raw feature views registered. Let's verify:

In [ ]:
# List all registered feature views
print("All Feature Views:")
fs.list_feature_views().show()

## 5. Engineered Feature Views

Raw features are rarely sufficient for strong model performance. Engineered features
capture behavioral patterns and derived signals that directly map to business logic.

We'll build two computed feature views:
1. **Customer Behavior Features** — RFM scores, category affinities from purchase history
2. **Product Popularity Features** — trending score, conversion rates from daily metrics

### 5.1 Customer Behavior Features (RFM + Category Affinity)

In [ ]:
# Compute RFM (Recency, Frequency, Monetary) scores and category affinities
# using Snowpark window functions on purchase history

customer_behavior_df = session.sql(f"""
    WITH purchase_stats AS (
        SELECT
            ph.CUSTOMER_ID,
            -- Recency: days since last purchase (lower = more recent)
            DATEDIFF('day', MAX(ph.PURCHASE_TS), CURRENT_TIMESTAMP()) AS DAYS_SINCE_LAST_PURCHASE,
            -- Frequency: total number of orders
            COUNT(DISTINCT DATE_TRUNC('day', ph.PURCHASE_TS)) AS PURCHASE_FREQUENCY,
            -- Monetary: total and average spend
            SUM(ph.TOTAL_AMOUNT) AS TOTAL_SPEND,
            AVG(ph.TOTAL_AMOUNT) AS AVG_TRANSACTION_VALUE,
            -- Behavioral signals
            COUNT(DISTINCT ph.PRODUCT_ID) AS UNIQUE_PRODUCTS_BOUGHT,
            AVG(ph.DISCOUNT_PCT) AS AVG_DISCOUNT_USED,
            STDDEV(ph.TOTAL_AMOUNT) AS SPEND_VOLATILITY,
            -- Time patterns
            COUNT(CASE WHEN DAYOFWEEK(ph.PURCHASE_TS) IN (0, 6) THEN 1 END)::FLOAT
                / NULLIF(COUNT(*), 0) AS WEEKEND_PURCHASE_RATIO
        FROM {DATABASE}.{SCHEMA_RAW}.PURCHASE_HISTORY ph
        GROUP BY ph.CUSTOMER_ID
    ),
    category_affinity AS (
        SELECT
            ph.CUSTOMER_ID,
            -- Top category share
            MAX(cat_count)::FLOAT / NULLIF(SUM(cat_count), 0) AS TOP_CATEGORY_CONCENTRATION,
            COUNT(DISTINCT p.CATEGORY) AS CATEGORIES_EXPLORED
        FROM (
            SELECT CUSTOMER_ID, PRODUCT_ID, COUNT(*) AS cat_count
            FROM {DATABASE}.{SCHEMA_RAW}.PURCHASE_HISTORY
            GROUP BY CUSTOMER_ID, PRODUCT_ID
        ) ph
        JOIN {DATABASE}.{SCHEMA_RAW}.PRODUCT_CATALOG p ON ph.PRODUCT_ID = p.PRODUCT_ID
        GROUP BY ph.CUSTOMER_ID
    )
    SELECT
        ps.CUSTOMER_ID,
        ps.DAYS_SINCE_LAST_PURCHASE,
        ps.PURCHASE_FREQUENCY,
        ps.TOTAL_SPEND,
        ps.AVG_TRANSACTION_VALUE,
        ps.UNIQUE_PRODUCTS_BOUGHT,
        ps.AVG_DISCOUNT_USED,
        ps.SPEND_VOLATILITY,
        ps.WEEKEND_PURCHASE_RATIO,
        -- RFM Score (normalized 0-100, higher = better customer)
        LEAST(100, GREATEST(0,
            (1.0 - LEAST(ps.DAYS_SINCE_LAST_PURCHASE, 365) / 365.0) * 35
            + LEAST(ps.PURCHASE_FREQUENCY, 50) / 50.0 * 35
            + LEAST(ps.TOTAL_SPEND, 10000) / 10000.0 * 30
        )) AS RFM_SCORE,
        ca.TOP_CATEGORY_CONCENTRATION,
        ca.CATEGORIES_EXPLORED
    FROM purchase_stats ps
    LEFT JOIN category_affinity ca ON ps.CUSTOMER_ID = ca.CUSTOMER_ID
""")

customer_behavior_fv = FeatureView(
    name="CUSTOMER_BEHAVIOR_FEATURES",
    entities=[customer_entity],
    feature_df=customer_behavior_df,
    desc="Engineered RFM scores, spending patterns, and category affinity metrics",
)
customer_behavior_fv = fs.register_feature_view(
    feature_view=customer_behavior_fv, version="v1", overwrite=True
)
print(f"Registered: {customer_behavior_fv.name}/{customer_behavior_fv.version}")
print(f"  Features: {len(customer_behavior_df.columns) - 1} engineered columns")

# Preview
customer_behavior_df.limit(5).show()

### 5.2 Product Popularity Features

In [ ]:
# Compute product engagement and conversion metrics
# from daily metrics and browse events

product_popularity_df = session.sql(f"""
    WITH product_engagement AS (
        SELECT
            p.PRODUCT_ID,
            -- Popularity from daily metrics
            AVG(dm.DAILY_VIEWS) AS AVG_DAILY_VIEWS,
            AVG(dm.DAILY_PURCHASES) AS AVG_DAILY_PURCHASES,
            AVG(dm.DAILY_CART_ADDS) AS AVG_DAILY_CART_ADDS,
            -- View-to-purchase conversion
            SUM(dm.DAILY_PURCHASES)::FLOAT / NULLIF(SUM(dm.DAILY_VIEWS), 0) AS VIEW_TO_PURCHASE_RATIO,
            -- Cart abandonment rate
            1.0 - (SUM(dm.DAILY_PURCHASES)::FLOAT / NULLIF(SUM(dm.DAILY_CART_ADDS), 0)) AS CART_ABANDONMENT_RATE,
            -- Trending: recent views vs historical average
            AVG(CASE WHEN dm.METRIC_DATE >= DATEADD('day', -7, CURRENT_DATE())
                     THEN dm.DAILY_VIEWS END)
            / NULLIF(AVG(dm.DAILY_VIEWS), 0) AS TRENDING_SCORE_7D,
            -- Revenue proxy: purchases * avg price
            SUM(dm.DAILY_PURCHASES) * AVG(p.PRICE) / NULLIF(COUNT(DISTINCT dm.METRIC_DATE), 0) AS AVG_DAILY_REVENUE
        FROM {DATABASE}.{SCHEMA_RAW}.PRODUCT_CATALOG p
        JOIN {DATABASE}.{SCHEMA_RAW}.PRODUCT_DAILY_METRICS dm ON p.PRODUCT_ID = dm.PRODUCT_ID
        GROUP BY p.PRODUCT_ID
    ),
    browse_stats AS (
        SELECT
            PRODUCT_ID,
            COUNT(DISTINCT CUSTOMER_ID) AS UNIQUE_BROWSERS,
            AVG(CASE WHEN EVENT_TYPE = 'wishlist' THEN 1 ELSE 0 END) AS WISHLIST_RATE
        FROM {DATABASE}.{SCHEMA_RAW}.BROWSE_EVENTS
        GROUP BY PRODUCT_ID
    )
    SELECT
        pe.PRODUCT_ID,
        pe.AVG_DAILY_VIEWS,
        pe.AVG_DAILY_PURCHASES,
        pe.VIEW_TO_PURCHASE_RATIO,
        pe.CART_ABANDONMENT_RATE,
        pe.TRENDING_SCORE_7D,
        pe.AVG_DAILY_REVENUE,
        bs.UNIQUE_BROWSERS,
        bs.WISHLIST_RATE,
        -- Composite popularity score
        LEAST(100, GREATEST(0,
            COALESCE(pe.TRENDING_SCORE_7D, 1.0) * 25
            + LEAST(pe.VIEW_TO_PURCHASE_RATIO * 100, 25)
            + LEAST(pe.AVG_DAILY_VIEWS / 100.0, 25)
            + COALESCE(bs.WISHLIST_RATE, 0) * 25
        )) AS POPULARITY_SCORE
    FROM product_engagement pe
    LEFT JOIN browse_stats bs ON pe.PRODUCT_ID = bs.PRODUCT_ID
""")

product_popularity_fv = FeatureView(
    name="PRODUCT_POPULARITY_FEATURES",
    entities=[product_entity],
    feature_df=product_popularity_df,
    desc="Product engagement metrics: conversion rates, trending score, popularity composite",
)
product_popularity_fv = fs.register_feature_view(
    feature_view=product_popularity_fv, version="v1", overwrite=True
)
print(f"Registered: {product_popularity_fv.name}/{product_popularity_fv.version}")
print(f"  Features: {len(product_popularity_df.columns) - 1} engineered columns")

# Preview
product_popularity_df.limit(5).show()

## 6. Generate Training Dataset

The Feature Store's `generate_dataset()` performs **point-in-time correct joins** —
ensuring each training example only uses features available at the time of the event.
This prevents data leakage.

**Spine concept:** The spine is the set of (entity_key, timestamp, label) tuples that
define what we're predicting. We build it from:
- **Positive samples** (PURCHASED=1): customer actually bought the product
- **Negative samples** (PURCHASED=0): customer viewed but did NOT buy the product

This gives us a balanced classification dataset for recommendation scoring.

In [ ]:
# Build the training spine
# Positive samples: actual purchases
# Negative samples: viewed/carted but not purchased (within same time window)

spine_df = session.sql(f"""
    WITH positives AS (
        -- Customer bought the product
        SELECT DISTINCT
            CUSTOMER_ID,
            PRODUCT_ID,
            PURCHASE_TS AS EVENT_TS,
            1 AS PURCHASED
        FROM {DATABASE}.{SCHEMA_RAW}.PURCHASE_HISTORY
    ),
    negatives AS (
        -- Customer browsed but did NOT purchase (negative sampling)
        SELECT
            b.CUSTOMER_ID,
            b.PRODUCT_ID,
            b.EVENT_TS,
            0 AS PURCHASED
        FROM {DATABASE}.{SCHEMA_RAW}.BROWSE_EVENTS b
        LEFT JOIN {DATABASE}.{SCHEMA_RAW}.PURCHASE_HISTORY ph
            ON b.CUSTOMER_ID = ph.CUSTOMER_ID
            AND b.PRODUCT_ID = ph.PRODUCT_ID
            AND ph.PURCHASE_TS BETWEEN b.EVENT_TS AND DATEADD('day', 7, b.EVENT_TS)
        WHERE ph.CUSTOMER_ID IS NULL  -- no purchase within 7 days of browsing
          AND b.EVENT_TYPE IN ('page_view', 'add_to_cart')  -- meaningful engagement
    ),
    sampled_negatives AS (
        SELECT * FROM negatives SAMPLE BERNOULLI (50)
    )
    -- Combine: all positives + sampled negatives
    SELECT * FROM positives
    UNION ALL
    SELECT * FROM sampled_negatives
""")

total_rows = spine_df.count()
positive_rate = spine_df.filter(F.col("PURCHASED") == 1).count() / total_rows
print(f"Training spine: {total_rows:,} samples")
print(f"Positive rate (purchased): {positive_rate:.2%}")
print(f"Negative rate (not purchased): {1 - positive_rate:.2%}")
spine_df.limit(10).show()

In [ ]:
# Generate training dataset with point-in-time correct feature joins
# The Feature Store automatically joins features based on entity keys and timestamps

# Retrieve registered feature views
customer_profile_fv = fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "v1")
product_catalog_fv = fs.get_feature_view("PRODUCT_CATALOG_FEATURES", "v1")
customer_behavior_fv = fs.get_feature_view("CUSTOMER_BEHAVIOR_FEATURES", "v1")
product_popularity_fv = fs.get_feature_view("PRODUCT_POPULARITY_FEATURES", "v1")

all_feature_views = [
    customer_profile_fv,
    product_catalog_fv,
    customer_behavior_fv,
    product_popularity_fv,
]

# Drop existing dataset if it exists (for re-runnability)
try:
    session.sql(f"DROP DATASET IF EXISTS {DATABASE}.{SCHEMA_FS}.ECOMMERCE_TRAINING_SET").collect()
except Exception:
    pass

# Generate the full training dataset
training_dataset = fs.generate_dataset(
    name=f"{DATABASE}.{SCHEMA_FS}.ECOMMERCE_TRAINING_SET",
    version="v1",
    spine_df=spine_df,
    features=all_feature_views,
    spine_timestamp_col="EVENT_TS",
    spine_label_cols=["PURCHASED"],
    desc="E-commerce purchase prediction training set with PIT-correct features",
)

training_df = training_dataset.read.to_snowpark_dataframe()
print(f"Training dataset generated: {training_df.count():,} rows, {len(training_df.columns)} columns")
print(f"\nColumns: {training_df.columns}")

In [ ]:
# Split train/test with random sampling (80/20)
# First materialize the full dataset, then split deterministically

training_df.write.mode("overwrite").save_as_table(
    f"{DATABASE}.{SCHEMA_FS}.TRAINING_MATERIALIZED"
)

# Use SQL to add a deterministic random column for splitting
session.sql(f"""
    CREATE OR REPLACE TABLE {DATABASE}.{SCHEMA_FS}.TRAINING_WITH_SPLIT AS
    SELECT *, ABS(HASH(CUSTOMER_ID, PRODUCT_ID, EVENT_TS)) % 100 AS _SPLIT_HASH
    FROM {DATABASE}.{SCHEMA_FS}.TRAINING_MATERIALIZED
""").collect()

split_table = session.table(f"{DATABASE}.{SCHEMA_FS}.TRAINING_WITH_SPLIT")
train_df = split_table.filter(F.col("_SPLIT_HASH") < 80).drop("_SPLIT_HASH")
test_df = split_table.filter(F.col("_SPLIT_HASH") >= 80).drop("_SPLIT_HASH")

train_count = train_df.count()
test_count = test_df.count()

print(f"Train set: {train_count:,} rows (~80%)")
print(f"Test set:  {test_count:,} rows (~20%)")

train_positive_rate = train_df.filter(F.col("PURCHASED") == 1).count() / train_count
test_positive_rate = test_df.filter(F.col("PURCHASED") == 1).count() / test_count
print(f"\nTrain purchase rate: {train_positive_rate:.2%}")
print(f"Test purchase rate:  {test_positive_rate:.2%}")

## 7. Train Recommendation Model

We train an XGBoost binary classifier to predict purchase probability.
The model is trained using Snowflake's distributed ML infrastructure —
data never leaves the platform.

After training, we register the model in the **Model Registry** for
versioned, governed access.

In [ ]:
# Define feature columns (exclude IDs, timestamps, and label)
exclude_cols = [
    "CUSTOMER_ID", "PRODUCT_ID", "EVENT_TS", "PURCHASED",
    "PURCHASE_TS",  # from raw purchase history FV
    # Categorical string columns (XGBoost handles natively in Snowflake ML)
    "GENDER", "STATE", "LOYALTY_TIER", "PREFERRED_CATEGORY",
    "CATEGORY", "SUBCATEGORY", "BRAND", "AGE_GROUP",
]

feature_cols = [c for c in training_df.columns if c not in exclude_cols]
label_col = "PURCHASED"

print(f"Feature columns ({len(feature_cols)}):")
for i in range(0, len(feature_cols), 4):
    print(f"  {', '.join(feature_cols[i:i+4])}")
print(f"\nLabel: {label_col}")

In [ ]:
# Ensure database context is set for the server-side training procedure
session.use_database(DATABASE)
session.use_schema(SCHEMA_MODELS)

# Drop columns that are entirely null (can't infer signature)
null_cols = [c for c in feature_cols if train_df.where(F.col(c).is_not_null()).count() == 0]
if null_cols:
    print(f"Dropping all-null columns: {null_cols}")
    feature_cols = [c for c in feature_cols if c not in null_cols]

# Train XGBoost binary classifier
model = XGBClassifier(
    input_cols=feature_cols,
    label_cols=[label_col],
    output_cols=["PREDICTED_PURCHASED"],
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=2.0,  # adjust for class imbalance
)

print("Training XGBoost model...")
model.fit(train_df)
print("Training complete.")

In [ ]:
# Evaluate on test set
predictions_df = model.predict(test_df)

auc = roc_auc_score(df=predictions_df, y_true_col_names="PURCHASED", y_score_col_names="PREDICTED_PURCHASED")
precision = precision_score(df=predictions_df, y_true_col_names="PURCHASED", y_pred_col_names="PREDICTED_PURCHASED")
recall = recall_score(df=predictions_df, y_true_col_names="PURCHASED", y_pred_col_names="PREDICTED_PURCHASED")
f1 = f1_score(df=predictions_df, y_true_col_names="PURCHASED", y_pred_col_names="PREDICTED_PURCHASED")

print("=" * 50)
print("MODEL EVALUATION: PRODUCT_RECOMMENDER")
print("=" * 50)
print(f"  AUC:       {auc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print("=" * 50)

In [ ]:
# Register model in the Model Registry
reg = Registry(
    session=session,
    database_name=DATABASE,
    schema_name=SCHEMA_MODELS,
)

mv = reg.log_model(
    model=model,
    model_name="PRODUCT_RECOMMENDER",
    metrics={"auc": auc, "precision": precision, "recall": recall},
    comment="XGBoost binary classifier for product purchase prediction",
    sample_input_data=train_df.select(feature_cols).limit(100),
)

print(f"Model registered: PRODUCT_RECOMMENDER/v1")
print(f"  AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
print(f"  Location: {DATABASE}.{SCHEMA_MODELS}.PRODUCT_RECOMMENDER")

## 7.5 Feature Set Experiments

Not all features improve model performance equally. Some may add noise or overfit.
We use **Snowflake ML Experiments** to systematically compare different feature
subsets and identify the most effective combination.

**Approach:**
1. Define candidate feature sets (e.g., profile-only, behavior-only, all features)
2. Train a model for each feature set under the same conditions
3. Log metrics to the experiment for structured comparison
4. Select the best-performing feature set for production deployment

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

# Create an experiment tracker and set up the experiment for feature set comparisons
experiment = ExperimentTracking(session, database_name=DATABASE, schema_name=SCHEMA_MODELS)
experiment.set_experiment("FEATURE_SET_COMPARISON")

print(f"Experiment created: {DATABASE}.{SCHEMA_MODELS}.FEATURE_SET_COMPARISON")

In [ ]:
# Define candidate feature sets to compare
# Each set represents a hypothesis about which features drive purchase prediction

# Numeric-only subsets from the training data
all_numeric_features = [c for c in feature_cols if c not in exclude_cols]

# Feature Set 1: Customer profile features only (demographics + spend history)
profile_features = [
    c for c in all_numeric_features
    if c in [
        "DAYS_SINCE_LAST_ORDER", "LIFETIME_SPEND", "LIFETIME_ORDERS", "AVG_ORDER_VALUE",
    ]
]

# Feature Set 2: Customer behavior features only (RFM + engagement patterns)
behavior_features = [
    c for c in all_numeric_features
    if c in [
        "DAYS_SINCE_LAST_PURCHASE", "PURCHASE_FREQUENCY", "TOTAL_SPEND",
        "AVG_TRANSACTION_VALUE", "UNIQUE_PRODUCTS_BOUGHT", "AVG_DISCOUNT_USED",
        "SPEND_VOLATILITY", "WEEKEND_PURCHASE_RATIO", "RFM_SCORE",
        "TOP_CATEGORY_CONCENTRATION", "CATEGORIES_EXPLORED",
    ]
]

# Feature Set 3: Product features only (catalog + popularity signals)
product_features = [
    c for c in all_numeric_features
    if c in [
        "PRICE", "AVG_RATING", "REVIEW_COUNT", "IN_STOCK",
        "AVG_DAILY_VIEWS", "AVG_DAILY_PURCHASES", "VIEW_TO_PURCHASE_RATIO",
        "CART_ABANDONMENT_RATE", "TRENDING_SCORE_7D", "AVG_DAILY_REVENUE",
        "UNIQUE_BROWSERS", "WISHLIST_RATE", "POPULARITY_SCORE",
    ]
]

# Feature Set 4: All numeric features combined
all_features = all_numeric_features

feature_sets = {
    "profile_only": profile_features,
    "behavior_only": behavior_features,
    "product_only": product_features,
    "all_features": all_features,
}

print("Candidate feature sets:")
for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")
    if len(feats) <= 6:
        print(f"    {feats}")

In [ ]:
# Train and evaluate a model for each feature set, logging results to the experiment

results = []

for set_name, feats in feature_sets.items():
    if not feats:
        print(f"Skipping {set_name}: no matching features in training data")
        continue

    print(f"\nTraining: {set_name} ({len(feats)} features)...")

    # Train with same hyperparameters to isolate feature impact
    exp_model = XGBClassifier(
        input_cols=feats,
        label_cols=[label_col],
        output_cols=["PREDICTED_PURCHASED"],
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=2.0,
    )
    exp_model.fit(train_df)

    # Evaluate
    preds = exp_model.predict(test_df)
    exp_auc = roc_auc_score(df=preds, y_true_col_names="PURCHASED", y_score_col_names="PREDICTED_PURCHASED")
    exp_precision = precision_score(df=preds, y_true_col_names="PURCHASED", y_pred_col_names="PREDICTED_PURCHASED")
    exp_recall = recall_score(df=preds, y_true_col_names="PURCHASED", y_pred_col_names="PREDICTED_PURCHASED")
    exp_f1 = f1_score(df=preds, y_true_col_names="PURCHASED", y_pred_col_names="PREDICTED_PURCHASED")

    # Log run to the experiment
    experiment.start_run(run_name=set_name)
    experiment.log_metric("auc", exp_auc)
    experiment.log_metric("precision", exp_precision)
    experiment.log_metric("recall", exp_recall)
    experiment.log_metric("f1", exp_f1)
    experiment.log_param("feature_set", set_name)
    experiment.log_param("n_features", len(feats))
    experiment.log_param("features", ", ".join(feats[:20]))
    experiment.end_run()

    results.append({
        "feature_set": set_name,
        "n_features": len(feats),
        "auc": exp_auc,
        "precision": exp_precision,
        "recall": exp_recall,
        "f1": exp_f1,
    })

    print(f"  AUC={exp_auc:.4f}  Precision={exp_precision:.4f}  Recall={exp_recall:.4f}  F1={exp_f1:.4f}")

print("\nAll runs logged to experiment.")

In [ ]:
# Compare experiment results side-by-side
results_pdf = pd.DataFrame(results).sort_values("auc", ascending=False)

print("=" * 70)
print("EXPERIMENT RESULTS: FEATURE SET COMPARISON")
print("=" * 70)
print(results_pdf.to_string(index=False, float_format="{:.4f}".format))
print("=" * 70)

best = results_pdf.iloc[0]
print(f"\nBest feature set: {best['feature_set']}")
print(f"  AUC: {best['auc']:.4f} | F1: {best['f1']:.4f} | Features: {best['n_features']}")

# Check if adding more features actually helps
if best["feature_set"] != "all_features":
    all_row = results_pdf[results_pdf["feature_set"] == "all_features"].iloc[0]
    print(f"\nNote: '{best['feature_set']}' outperforms 'all_features' (AUC {all_row['auc']:.4f}).")
    print("  Consider dropping noisy features to reduce overfitting and serving cost.")
else:
    print("\n  All features combined yields the best performance.")
    print("  Consider using feature importance to prune low-value features for serving efficiency.")

## 8. Enable Online Serving

Online serving creates **hybrid table-backed replicas** of feature views for
sub-second point lookups. The `target_lag` parameter controls how fresh the
online copy stays relative to the offline (source) data.

This enables real-time recommendation systems to fetch the latest customer
and product features without querying the full warehouse.

In [ ]:
# Enable online serving on key feature views
# This creates hybrid tables that sync from the offline store

# Customer profile — relatively static, 1 minute lag is fine
customer_profile_fv = fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "v1")
fs.update_feature_view(
    name="CUSTOMER_PROFILE_FEATURES",
    version="v1",
    online_config=OnlineConfig(enable=True, target_lag="1 minute"),
)
print("Online enabled: CUSTOMER_PROFILE_FEATURES (target_lag=1 minute)")

# Customer behavior — changes with each purchase, fresher lag
fs.update_feature_view(
    name="CUSTOMER_BEHAVIOR_FEATURES",
    version="v1",
    online_config=OnlineConfig(enable=True, target_lag="30 seconds"),
)
print("Online enabled: CUSTOMER_BEHAVIOR_FEATURES (target_lag=30 seconds)")

# Product popularity — high-velocity metrics, keep very fresh
fs.update_feature_view(
    name="PRODUCT_POPULARITY_FEATURES",
    version="v1",
    online_config=OnlineConfig(enable=True, target_lag="30 seconds"),
)
print("Online enabled: PRODUCT_POPULARITY_FEATURES (target_lag=30 seconds)")

print("\nOnline serving configured for 3 feature views.")
print("Hybrid tables will sync automatically based on target_lag.")

## 9. Real-Time Feature Retrieval

With online serving enabled, we can fetch features with sub-second latency
using `StoreType.ONLINE`. This simulates what a production recommendation
service would do:

1. Receive a request: "What should we recommend to customer X?"
2. Fetch the customer's latest features from the online store
3. Score candidate products using the registered model
4. Return ranked recommendations

In [ ]:
# Sample customer IDs for demonstration
sample_customers = session.sql(f"""
    SELECT CUSTOMER_ID FROM {DATABASE}.{SCHEMA_RAW}.CUSTOMER_PROFILE
    ORDER BY LIFETIME_SPEND DESC
    LIMIT 3
""").collect()
sample_ids = [row["CUSTOMER_ID"] for row in sample_customers]
print(f"Sample customers (top spenders): {sample_ids}")

# Fetch features from ONLINE store (sub-second)
customer_behavior_fv = fs.get_feature_view("CUSTOMER_BEHAVIOR_FEATURES", "v1")

print("\n--- Online Feature Retrieval ---")
start = time.time()
try:
    online_features = fs.read_feature_view(
        feature_view=customer_behavior_fv,
        keys=[[cid] for cid in sample_ids],
        feature_names=[
            "RFM_SCORE", "DAYS_SINCE_LAST_PURCHASE", "PURCHASE_FREQUENCY",
            "TOTAL_SPEND", "TOP_CATEGORY_CONCENTRATION",
        ],
        store_type=StoreType.ONLINE,
    )
    online_latency = (time.time() - start) * 1000
    print(f"Online latency: {online_latency:.1f} ms")
    online_features.show()
except Exception as e:
    if "not refreshed yet" in str(e):
        print(f"Online store not yet ready (initial sync in progress).")
        print("This is expected right after enabling online serving.")
        print("The hybrid table will be ready shortly.")
    else:
        raise

# Compare with OFFLINE store (warehouse query)
print("\n--- Offline Feature Retrieval ---")
start = time.time()
offline_features = fs.read_feature_view(
    feature_view=customer_behavior_fv,
    keys=[[cid] for cid in sample_ids],
    feature_names=[
        "RFM_SCORE", "DAYS_SINCE_LAST_PURCHASE", "PURCHASE_FREQUENCY",
        "TOTAL_SPEND", "TOP_CATEGORY_CONCENTRATION",
    ],
)
offline_latency = (time.time() - start) * 1000
print(f"Offline latency: {offline_latency:.1f} ms")
offline_features.show()

In [ ]:
# Full flow: fetch features → score with model
# Simulates a real-time recommendation request

target_customer = sample_ids[0]
print(f"Scoring recommendations for customer: {target_customer}")

# Get candidate products
candidate_products = session.sql(f"""
    SELECT PRODUCT_ID FROM {DATABASE}.{SCHEMA_RAW}.PRODUCT_CATALOG
    LIMIT 20
""").collect()
product_ids = [row["PRODUCT_ID"] for row in candidate_products]

# Fetch product popularity features
product_popularity_fv = fs.get_feature_view("PRODUCT_POPULARITY_FEATURES", "v1")
try:
    product_features = fs.read_feature_view(
        feature_view=product_popularity_fv,
        keys=[[pid] for pid in product_ids],
        feature_names=["POPULARITY_SCORE", "VIEW_TO_PURCHASE_RATIO", "TRENDING_SCORE_7D"],
        store_type=StoreType.ONLINE,
    )
    print(f"\nFetched features for {len(product_ids)} candidate products (ONLINE)")
    print(f"\nTop products by popularity score:")
    product_features.order_by(F.col("POPULARITY_SCORE").desc()).limit(5).show()
except Exception as e:
    if "not refreshed yet" in str(e):
        print("Online store not yet refreshed. Falling back to offline store.")
        product_features = fs.read_feature_view(
            feature_view=product_popularity_fv,
            keys=[[pid] for pid in product_ids],
            feature_names=["POPULARITY_SCORE", "VIEW_TO_PURCHASE_RATIO", "TRENDING_SCORE_7D"],
        )
        print(f"\nFetched features for {len(product_ids)} candidate products (OFFLINE)")
        print(f"\nTop products by popularity score:")
        product_features.order_by(F.col("POPULARITY_SCORE").desc()).limit(5).show()
    else:
        raise

# Fetch customer features
customer_profile_fv = fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "v1")
try:
    cust_features = fs.read_feature_view(
        feature_view=customer_profile_fv,
        keys=[[target_customer]],
        feature_names=["LIFETIME_SPEND", "AVG_ORDER_VALUE", "LOYALTY_TIER"],
        store_type=StoreType.ONLINE,
    )
    print(f"\nCustomer profile (ONLINE):")
    cust_features.show()
except Exception as e:
    if "not refreshed yet" in str(e):
        cust_features = fs.read_feature_view(
            feature_view=customer_profile_fv,
            keys=[[target_customer]],
            feature_names=["LIFETIME_SPEND", "AVG_ORDER_VALUE", "LOYALTY_TIER"],
        )
        print(f"\nCustomer profile (OFFLINE fallback):")
        cust_features.show()
    else:
        raise

## 10. Deploy Model with Online Feature Integration

The `feature_sources_per_function` parameter connects the model service directly
to the online Feature Store. At inference time:

1. Client sends only entity keys (e.g., `CUSTOMER_ID`)
2. The service **automatically fetches** the latest features from the online store
3. The model scores with those features
4. Client receives predictions

This eliminates the need for clients to assemble feature vectors — reducing
latency, complexity, and the risk of training/serving skew.

In [ ]:
# Model serving with Feature Store integration
# In production, you would deploy the model as a service that auto-fetches features

customer_behavior_fv = fs.get_feature_view("CUSTOMER_BEHAVIOR_FEATURES", "v1")
mv = reg.get_model("PRODUCT_RECOMMENDER").version("v1")

print("=== Model Serving Architecture ===")
print(f"\nModel: PRODUCT_RECOMMENDER/v1")
print(f"Feature Source: CUSTOMER_BEHAVIOR_FEATURES/v1 (online store)")
print(f"\nInference flow:")
print(f"  1. Client sends: CUSTOMER_ID, PRODUCT_ID")
print(f"  2. Service auto-fetches: RFM_SCORE, PURCHASE_FREQUENCY, etc.")
print(f"  3. Model scores and returns: PREDICTED_PURCHASED")
print(f"\nTo deploy as a service (requires compute pool):")
print(f'  mv.create_service(')
print(f'      service_name="{DATABASE}.{SCHEMA_SCORING}.RECOMMENDER_SERVICE",')
print(f'      service_compute_pool="<YOUR_COMPUTE_POOL>",')
print(f'      image_build_compute_pool="<YOUR_COMPUTE_POOL>",')
print(f'      ingress_enabled=True,')
print(f'  )')

Demonstrate inference — client only sends entity keys
The service handles feature lookup automatically

Build a simple inference request with just entity keys
```py
inference_request = session.create_dataframe(
    [[sample_ids[0]], [sample_ids[1]], [sample_ids[2]]],
    schema=["CUSTOMER_ID"],
)

print("Inference request (entity keys only):")
inference_request.show()
```
In production, this would call the service:
```py
scored = mv.run(inference_request, function_name="predict", service_name=SERVICE_NAME)
scored.show()


print("\nIn production, call:")
print(f'  mv.run(request_df, function_name="predict", service_name="{SERVICE_NAME}")')
print("\nThe service automatically:")
print("  - Looks up CUSTOMER_BEHAVIOR_FEATURES from online store")
print("  - Joins features to the request")
print("  - Runs the model")
print("  - Returns predictions")
```

## 11. Monitoring & Operations

Production feature stores need monitoring for:
- **Refresh health** — are online tables staying in sync?
- **Cost tracking** — what's the compute cost of maintaining online features?
- **Feature drift** — have feature distributions shifted from training time?

In [ ]:
# Check online refresh history
customer_behavior_fv = fs.get_feature_view("CUSTOMER_BEHAVIOR_FEATURES", "v1")

print("=== Online Refresh History: CUSTOMER_BEHAVIOR_FEATURES ===")
refresh_history = fs.get_refresh_history(
    feature_view=customer_behavior_fv,
    store_type=StoreType.ONLINE,
)
refresh_history.show()

print("\n=== Online Refresh History: PRODUCT_POPULARITY_FEATURES ===")
product_popularity_fv = fs.get_feature_view("PRODUCT_POPULARITY_FEATURES", "v1")
refresh_history_prod = fs.get_refresh_history(
    feature_view=product_popularity_fv,
    store_type=StoreType.ONLINE,
)
refresh_history_prod.show()

In [ ]:
# Cost monitoring: track compute used by feature store operations
# Note: Static feature views don't use dynamic table refreshes.
# This query shows warehouse usage for the Feature Store schema.

cost_df = session.sql(f"""
    SELECT
        QUERY_TYPE,
        COUNT(*) AS QUERY_COUNT,
        SUM(TOTAL_ELAPSED_TIME) / 1000 AS TOTAL_ELAPSED_SEC,
        AVG(TOTAL_ELAPSED_TIME) / 1000 AS AVG_ELAPSED_SEC,
        SUM(CREDITS_USED_CLOUD_SERVICES) AS CLOUD_CREDITS
    FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
        RESULT_LIMIT => 100
    ))
    WHERE DATABASE_NAME = '{DATABASE}'
      AND SCHEMA_NAME = '{SCHEMA_FS}'
    GROUP BY QUERY_TYPE
    ORDER BY TOTAL_ELAPSED_SEC DESC
""")

print("=== Feature Store Query Cost Summary ===")
cost_df.show()

In [ ]:
# Feature drift detection: compare current distributions vs training time
# This helps detect when features have shifted and the model may need retraining

print("=== Feature Distribution Drift Check ===")
print("Comparing current feature stats vs training set stats\n")

# Current distribution of key features (from the feature view)
customer_behavior_fv = fs.get_feature_view("CUSTOMER_BEHAVIOR_FEATURES", "v1")
current_df = fs.read_feature_view(feature_view=customer_behavior_fv)

current_stats = current_df.select(
    F.lit("CURRENT").alias("PERIOD"),
    F.avg("RFM_SCORE").alias("AVG_RFM_SCORE"),
    F.stddev("RFM_SCORE").alias("STD_RFM_SCORE"),
    F.avg("PURCHASE_FREQUENCY").alias("AVG_PURCHASE_FREQ"),
    F.avg("TOTAL_SPEND").alias("AVG_TOTAL_SPEND"),
    F.avg("DAYS_SINCE_LAST_PURCHASE").alias("AVG_RECENCY"),
)

# Training-time distribution (from the materialized training data)
training_stats = train_df.select(
    F.lit("TRAINING").alias("PERIOD"),
    F.avg("RFM_SCORE").alias("AVG_RFM_SCORE"),
    F.stddev("RFM_SCORE").alias("STD_RFM_SCORE"),
    F.avg("PURCHASE_FREQUENCY").alias("AVG_PURCHASE_FREQ"),
    F.avg("TOTAL_SPEND").alias("AVG_TOTAL_SPEND"),
    F.avg("DAYS_SINCE_LAST_PURCHASE").alias("AVG_RECENCY"),
)

current_stats.union_all(training_stats).show()

print("\nIf distributions diverge significantly, consider:")
print("  1. Retraining the model on recent data")
print("  2. Investigating upstream data quality issues")
print("  3. Adjusting feature engineering logic")

## 12. Summary

| Step | What We Did | Snowflake Capability |
|------|-------------|---------------------|
| Entity & FS Setup | Defined CUSTOMER and PRODUCT entities, initialized Feature Store | Feature Store (Entity, FeatureStore) |
| Raw Feature Views | Registered 3 source tables as governed feature views | Feature Store (FeatureView) |
| Engineered Features | Built RFM scores, category affinity, product popularity composites | Snowpark SQL, Feature Store |
| Training Dataset | Point-in-time correct joins with positive/negative sampling | Feature Store (generate_dataset) |
| Model Training | XGBoost binary classifier for purchase prediction | Snowpark ML (XGBClassifier) |
| Model Registry | Versioned model with tracked metrics | Model Registry (log_model) |
| Online Serving | Sub-second feature lookups via hybrid tables | Online Feature Store (OnlineConfig) |
| Model Deployment | Auto-fetch features at inference via feature_sources_per_function | Model Services + Feature Store |
| Monitoring | Refresh history, cost tracking, drift detection | Feature Store ops, INFORMATION_SCHEMA |

**Key takeaway:** Features are defined once and served both offline (training) and
online (real-time inference). The model service automatically fetches fresh features
at prediction time — eliminating training/serving skew and reducing client complexity.